# 2. Coulomb blockade and many-body states

**Learning goals.** In this tutorial you will:

- construct the spinful Anderson model;
- connect its single-particle input to four many-body states;
- understand how charging energy suppresses transport; and
- interpret state probabilities and spin-resolved lead channels.

We continue to use $\hbar=k_\mathrm{B}=|e|=1$ and the Pauli master equation.

## From one level to an interacting orbital

A spin-degenerate orbital has states $\uparrow$ and $\downarrow$. Its Hamiltonian is

$$H_\mathrm{dot}=\varepsilon(n_\uparrow+n_\downarrow)+U n_\uparrow n_\downarrow.$$

The many-body states are $|0\rangle$, $|\uparrow\rangle$, $|\downarrow\rangle$, and $|\uparrow\downarrow\rangle$, with energies $0$, $\varepsilon$, $\varepsilon$, and $2\varepsilon+U$. The interaction $U$ is the extra energy needed to add the second electron.

At the particle-hole-symmetric point $\varepsilon=-U/2$, the singly occupied states have the lowest energy. If a small source-drain bias cannot pay the energy needed to change the charge, sequential transport is suppressed: this is Coulomb blockade.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import qmeq

## Build the Anderson model

QmeQ represents the two physical reservoirs by four lead channels here: L$\uparrow$, R$\uparrow$, L$\downarrow$, and R$\downarrow$. A physical left current is therefore the sum of channels 0 and 2.

In [ ]:
U = 4.0
epsilon = -U / 2
temperature = 0.2
gamma = 0.1
bandwidth = 40.0
tunnel_amplitude = np.sqrt(gamma / (2 * np.pi))

def make_anderson_system(bias=0.0):
    return qmeq.Builder(
        nsingle=2,
        hsingle={(0, 0): epsilon, (1, 1): epsilon},
        coulomb={(0, 1, 1, 0): U},
        nleads=4,
        tleads={
            (0, 0): tunnel_amplitude,  # L up
            (1, 0): tunnel_amplitude,  # R up
            (2, 1): tunnel_amplitude,  # L down
            (3, 1): tunnel_amplitude,  # R down
        },
        mulst={0: bias / 2, 1: -bias / 2, 2: bias / 2, 3: -bias / 2},
        tlst={0: temperature, 1: temperature, 2: temperature, 3: temperature},
        dband=bandwidth,
        kerntype="Pauli",
    )

system = make_anderson_system()

Calling `solve(masterq=False)` diagonalizes the dot Hamiltonian without solving the transport problem; `system.Ea` then holds the many-body energies. Before that first call, `system.Ea` is still all zeros. `system.si.get_state(b)` gives the occupation-number representation of state $b$; in this convention the two entries refer to the spin-up and spin-down single-particle states.

In [ ]:
system.solve(masterq=False)

for state_index, energy in enumerate(system.Ea):
    occupation = system.si.get_state(state_index)
    charge = sum(occupation)
    print(f"state {state_index}: occupation={occupation}, charge={charge}, energy={energy:.2f}")

assert np.isclose(min(system.Ea), epsilon)
assert np.isclose(max(system.Ea), 2 * epsilon + U)

The numerical ordering of the two singly occupied states is an implementation detail. Their occupation vectors and energies carry the physical meaning.

**Prediction before calculating.** At zero bias the current must vanish. Near $\varepsilon=-U/2$ and at low temperature, almost all probability should lie in the two singly occupied states.

In [ ]:
system.solve()

for state_index, probability in enumerate(system.phi0):
    print(system.si.get_state(state_index), f"P = {probability:.6f}")

print("spin-resolved currents [L-up, R-up, L-down, R-down]:")
print(system.current)

assert np.isclose(np.sum(system.phi0), 1.0)
assert np.all(system.phi0 >= -1e-12)
assert np.allclose(system.current, 0.0, atol=1e-12)
assert np.isclose(system.phi0[1] + system.phi0[2], 1.0, atol=1e-4)

## Opening and closing the transport window

We now sweep the bias voltage while keeping the gate at the center of the singly occupied region. The chemical potentials are $\mu_L=V/2$ and $\mu_R=-V/2$.

For small $V$, neither an empty nor a doubly occupied intermediate state is energetically accessible by sequential tunnelling, so the current is thermally suppressed. Once the bias becomes comparable to the addition energy, current can flow.

In [ ]:
bias_values = np.linspace(0.0, 8.0, 81)
left_current = np.empty_like(bias_values)
mean_charge = np.empty_like(bias_values)
charges = np.array([sum(system.si.get_state(b)) for b in range(len(system.Ea))])

for index, bias in enumerate(bias_values):
    system.change(mulst={0: bias / 2, 1: -bias / 2, 2: bias / 2, 3: -bias / 2})
    system.solve(qdq=False)
    left_current[index] = system.current[0] + system.current[2]
    mean_charge[index] = np.dot(charges, system.phi0)
    assert np.isclose(np.sum(system.current), 0.0, atol=1e-10)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(bias_values / U, left_current / gamma)
axes[0].set(xlabel="$V/U$", ylabel="$I_L/\\Gamma$", title="Coulomb-blockade threshold")
axes[1].plot(bias_values / U, mean_charge)
axes[1].set(xlabel="$V/U$", ylabel="$\\langle N\\rangle$", title="Mean dot charge")
fig.tight_layout()

The rounded onset is physical: finite temperature smooths the reservoir Fermi surfaces. The exact curve also depends on the coupling asymmetry and on where the level lies relative to the bias window.

## Scope of the result

This calculation describes sequential changes between well-defined many-body states. Deep inside the blockade region, higher-order cotunnelling can produce a small current that the Pauli equation omits. A later tutorial treats that regime with second-order approaches.

## Exercises

1. Increase `temperature` from `0.2` to `0.5`. Predict and then observe how the threshold changes.
2. Change the gate to `epsilon = 0`. Which charge states become degenerate, and why does low-bias transport reappear?
3. Make the right coupling half as large as the left coupling. Which barrier limits the stationary current?